# Employee Attrition Predictor: Machine Learning para Retención de Talento

**Autor:** Eduardo Moraga | [LinkedIn](https://www.linkedin.com/in/eduardomoragaortega/) | [GitHub](https://github.com/EduardoMoraga)

---

## Contexto de Negocio

La **rotación de personal** es uno de los costos ocultos más grandes en cualquier organización:
- Reemplazar un empleado cuesta entre **50% y 200%** de su salario anual
- La pérdida de conocimiento institucional impacta la productividad del equipo
- Identificar empleados en riesgo de abandono **antes** de que renuncien permite intervenciones proactivas

Este proyecto construye un modelo predictivo de abandono laboral usando **XGBoost** con explicabilidad vía **SHAP**, transformándolo de una caja negra a una herramienta de decisión para RRHH.

### Variables del Dataset
| Variable | Descripción |
|----------|-------------|
| satisfaccion | Nivel de satisfacción laboral (0-1) |
| evaluacion | Última evaluación de desempeño (0-1) |
| numero_proyecto | Cantidad de proyectos asignados |
| horas_mensuales | Horas trabajadas por mes |
| antiguedad_empresa | Años en la empresa |
| accidentes | Accidentes laborales (0/1) |
| promocionado | Promoción en últimos 5 años (0/1) |
| departamento | Área funcional |
| nivel_salario | Bajo / Intermedio / Alto |
| **abandono** | **Variable target: Si / No** |

## 1. Setup y Carga de Datos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (classification_report, confusion_matrix, 
                             roc_auc_score, roc_curve, precision_recall_curve)
from xgboost import XGBClassifier
import shap
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

SAVE_FIGS = True
FIG_PATH = '../outputs/figures/'

In [ ]:
# Cargar y parsear el dataset
raw = pd.read_excel('../data/empleados.xlsx')
col = raw.columns[0]
headers = [h.strip() for h in col.split(',')]
df = raw[col].str.split(',', expand=True)
df.columns = headers

# Limpiar espacios y convertir tipos
for c in df.columns:
    df[c] = df[c].str.strip()

numeric_cols = ['satisfaccion', 'evaluacion', 'numero_proyecto', 
                'horas_mensuales', 'antiguedad_empresa', 'accidentes', 'promocionado']
for c in numeric_cols:
    df[c] = pd.to_numeric(df[c])

print(f"Dataset: {df.shape[0]:,} empleados | {df.shape[1]} variables")
print(f"Tasa de abandono: {(df['abandono'] == 'Si').mean():.1%}")
df.head()

## 2. Análisis Exploratorio (EDA)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Distribución de Variables por Estado de Abandono', fontsize=15, fontweight='bold')

for ax, col in zip(axes.flatten(), numeric_cols[:6]):
    for label, color in [('Si', '#e74c3c'), ('No', '#2ecc71')]:
        subset = df[df['abandono'] == label][col]
        ax.hist(subset, bins=30, alpha=0.6, label=f'Abandono: {label}', color=color, density=True)
    ax.set_title(col, fontweight='bold')
    ax.legend()

plt.tight_layout()
if SAVE_FIGS: plt.savefig(f'{FIG_PATH}eda_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Tasa de abandono por departamento y nivel salarial
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Por departamento
dept_attrition = df.groupby('departamento')['abandono'].apply(lambda x: (x == 'Si').mean()).sort_values(ascending=True)
colors_dept = ['#e74c3c' if v > 0.25 else '#f39c12' if v > 0.2 else '#2ecc71' for v in dept_attrition.values]
dept_attrition.plot(kind='barh', ax=axes[0], color=colors_dept)
axes[0].set_title('Tasa de Abandono por Departamento', fontweight='bold')
axes[0].set_xlabel('Tasa de Abandono')
axes[0].axvline(df['abandono'].apply(lambda x: x == 'Si').mean(), color='gray', linestyle='--', label='Promedio')
axes[0].legend()

# Por nivel salarial
salary_attrition = df.groupby('nivel_salario')['abandono'].apply(lambda x: (x == 'Si').mean())
salary_order = ['bajo', 'intermedio', 'alto']
salary_attrition = salary_attrition.reindex([s for s in salary_order if s in salary_attrition.index])
colors_sal = ['#e74c3c', '#f39c12', '#2ecc71']
salary_attrition.plot(kind='bar', ax=axes[1], color=colors_sal[:len(salary_attrition)])
axes[1].set_title('Tasa de Abandono por Nivel Salarial', fontweight='bold')
axes[1].set_ylabel('Tasa de Abandono')
axes[1].tick_params(axis='x', rotation=0)

plt.tight_layout()
if SAVE_FIGS: plt.savefig(f'{FIG_PATH}attrition_by_category.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Matriz de correlación
df_numeric = df[numeric_cols].copy()
df_numeric['abandono'] = (df['abandono'] == 'Si').astype(int)

fig, ax = plt.subplots(figsize=(10, 8))
corr = df_numeric.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, cmap='RdBu_r', center=0, 
            fmt='.2f', ax=ax, linewidths=0.5, vmin=-1, vmax=1)
ax.set_title('Matriz de Correlación', fontsize=14, fontweight='bold')

plt.tight_layout()
if SAVE_FIGS: plt.savefig(f'{FIG_PATH}correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Feature Engineering y Preprocesamiento

In [ ]:
# Encoding de variables categóricas
df_model = df.copy()

# Label encode target
df_model['abandono'] = (df_model['abandono'] == 'Si').astype(int)

# One-hot encode departamento
dept_dummies = pd.get_dummies(df_model['departamento'], prefix='dept', drop_first=True)

# Ordinal encode nivel_salario
salary_map = {'bajo': 0, 'intermedio': 1, 'alto': 2}
df_model['nivel_salario_ord'] = df_model['nivel_salario'].map(salary_map)

# Feature engineering
df_model['satisfaccion_x_evaluacion'] = df_model['satisfaccion'] * df_model['evaluacion']
df_model['horas_por_proyecto'] = df_model['horas_mensuales'] / df_model['numero_proyecto'].clip(lower=1)
df_model['overwork'] = (df_model['horas_mensuales'] > 240).astype(int)

# Preparar features finales
feature_cols = numeric_cols + ['nivel_salario_ord', 'satisfaccion_x_evaluacion', 
                                'horas_por_proyecto', 'overwork']
X = pd.concat([df_model[feature_cols], dept_dummies], axis=1)
y = df_model['abandono']

print(f"Features: {X.shape[1]} | Target balance: {y.value_counts().to_dict()}")
X.head()

## 4. Modelado con XGBoost

In [ ]:
# Train/test split estratificado
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# XGBoost con manejo de desbalance
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    random_state=42,
    eval_metric='logloss',
    early_stopping_rounds=20
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=False
)

print(f"Best iteration: {model.best_iteration}")
print(f"Best score: {model.best_score:.4f}")

In [ ]:
# Cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.05,
                  scale_pos_weight=scale_pos_weight, random_state=42, eval_metric='logloss'),
    X, y, cv=cv, scoring='roc_auc'
)
print(f"Cross-Validation AUC-ROC: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

## 5. Evaluación del Modelo

In [ ]:
# Predicciones
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=['No Abandona', 'Abandona']))
print(f"AUC-ROC: {roc_auc_score(y_test, y_prob):.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['No Abandona', 'Abandona'], yticklabels=['No Abandona', 'Abandona'])
axes[0].set_title('Matriz de Confusión', fontweight='bold')
axes[0].set_ylabel('Real')
axes[0].set_xlabel('Predicho')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)
axes[1].plot(fpr, tpr, color='#2196F3', lw=2, label=f'XGBoost (AUC = {auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', alpha=0.3)
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('Curva ROC', fontweight='bold')
axes[1].legend()

# Precision-Recall
prec, rec, _ = precision_recall_curve(y_test, y_prob)
axes[2].plot(rec, prec, color='#4CAF50', lw=2)
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].set_title('Curva Precision-Recall', fontweight='bold')
axes[2].axhline((y_test == 1).mean(), color='gray', linestyle='--', label='Baseline')
axes[2].legend()

plt.tight_layout()
if SAVE_FIGS: plt.savefig(f'{FIG_PATH}model_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Explicabilidad con SHAP

SHAP (SHapley Additive exPlanations) transforma el modelo en una herramienta transparente para RRHH. Cada predicción se explica mostrando **cuánto contribuye cada variable** a la probabilidad de abandono.

In [ ]:
# SHAP values
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Summary plot - importancia global
fig, ax = plt.subplots(figsize=(12, 8))
shap.summary_plot(shap_values, X_test, plot_type='bar', show=False, max_display=15)
plt.title('Importancia Global de Variables (SHAP)', fontsize=14, fontweight='bold')
plt.tight_layout()
if SAVE_FIGS: plt.savefig(f'{FIG_PATH}shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# SHAP Beeswarm - impacto direccional
fig, ax = plt.subplots(figsize=(12, 8))
shap.summary_plot(shap_values, X_test, show=False, max_display=15)
plt.title('Impacto Direccional de Variables en Abandono (SHAP)', fontsize=14, fontweight='bold')
plt.tight_layout()
if SAVE_FIGS: plt.savefig(f'{FIG_PATH}shap_beeswarm.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Ejemplo individual: empleado con alto riesgo
high_risk_idx = y_prob.argsort()[-1]
print(f"Empleado con mayor riesgo de abandono (prob: {y_prob[high_risk_idx]:.1%})")

fig, ax = plt.subplots(figsize=(14, 4))
shap.force_plot(explainer.expected_value, shap_values[high_risk_idx], 
                X_test.iloc[high_risk_idx], matplotlib=True, show=False)
plt.title('Explicación Individual - Empleado de Mayor Riesgo', fontsize=12, fontweight='bold')
plt.tight_layout()
if SAVE_FIGS: plt.savefig(f'{FIG_PATH}shap_individual.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Perfiles de Riesgo y Recomendaciones para RRHH

In [ ]:
# Crear perfiles de riesgo
df_results = X_test.copy()
df_results['prob_abandono'] = y_prob
df_results['riesgo'] = pd.cut(y_prob, bins=[0, 0.3, 0.6, 1.0], labels=['Bajo', 'Medio', 'Alto'])

print("=" * 70)
print("DISTRIBUCIÓN DE RIESGO EN LA ORGANIZACIÓN")
print("=" * 70)
risk_dist = df_results['riesgo'].value_counts()
for risk, count in risk_dist.items():
    pct = count / len(df_results) * 100
    print(f"  {risk:>5}: {count:,} empleados ({pct:.1f}%)")

print(f"\n{'=' * 70}")
print("PERFIL PROMEDIO POR NIVEL DE RIESGO")
print("=" * 70)
profile = df_results.groupby('riesgo')[['satisfaccion', 'evaluacion', 'numero_proyecto', 
                                         'horas_mensuales', 'antiguedad_empresa']].mean().round(2)
print(profile.to_string())

print(f"\n{'=' * 70}")
print("RECOMENDACIONES PARA RRHH")
print("=" * 70)
print("""
1. GRUPO ALTO RIESGO:
   - Agendar entrevistas de retención inmediatas
   - Evaluar carga de trabajo (proyectos y horas)
   - Revisar compensación vs. mercado

2. GRUPO MEDIO RIESGO:
   - Monitoreo trimestral de satisfacción
   - Plan de desarrollo profesional personalizado
   - Evaluar oportunidades de promoción

3. PREVENCIÓN GENERAL:
   - Controlar que ningún empleado supere 240 hrs/mes
   - Distribuír proyectos de forma equitativa (evitar >5 simultáneos)
   - Revisar salarios del segmento 'bajo' con >3 años de antigüedad
""")

---

## Próximos Pasos

- **Deployment**: API REST con FastAPI para scoring en tiempo real
- **Dashboard**: Panel de monitoreo de riesgo por departamento
- **Feedback loop**: Incorporar datos de salida real para reentrenar
- **Cost-sensitive learning**: Ponderar el costo real de cada tipo de error

---

*Proyecto desarrollado por [Eduardo Moraga](https://www.linkedin.com/in/eduardomoragaortega/) | Trade Marketing × Data Science*